In [4]:
text = """# BikeStore Database Knowledge Base

## Table: sales.customers

### Description

Stores customer information for people who purchase products from the store.

### Business Purpose

Represents customers and their personal/contact information. Used to analyze customer behavior, purchase history, customer segmentation, and geographic distribution.

### Columns

* customer_id: Unique customer identifier.
* first_name: Customer first name.
* last_name: Customer last name.
* phone: Customer phone number.
* email: Customer email address.
* street: Street address.
* city: Customer city.
* state: Customer state.
* zip_code: Postal code.

### Business Keywords

customer, buyer, client, consumer, shopper, user, customer profile, customer information

### Typical Questions

* Who are the top customers by sales?
* How many customers do we have?
* Which cities have the most customers?
* What customers have not placed orders recently?

## Table: sales.orders

### Description

Stores customer orders and tracks the order lifecycle.

### Business Purpose

Represents sales transactions made by customers. Used for sales analysis, order tracking, revenue reporting, and customer purchase history.

### Columns

* order_id: Unique order identifier.
* customer_id: Customer who placed the order.
* order_status: Status of the order.
* order_date: Date order was created.
* required_date: Requested delivery date.
* shipped_date: Date order was shipped.
* store_id: Store responsible for the order.
* staff_id: Employee responsible for the order.

### Relationships

* customer_id -> sales.customers.customer_id
* store_id -> sales.stores.store_id
* staff_id -> sales.staffs.staff_id

### Business Keywords

order, purchase, sale, transaction, revenue, customer order, sales order

### Typical Questions

* How many orders were placed last month?
* What is the monthly sales trend?
* Which customers placed the most orders?
* How many orders were shipped late?

## Table: sales.order_items

### Description

Stores individual products contained in each order.

### Business Purpose

Represents order line items. Used to calculate revenue, product sales, quantities sold, discounts, and product performance.

### Columns

* order_id: Associated order.
* item_id: Line item identifier.
* product_id: Product sold.
* quantity: Quantity sold.
* list_price: Product price at sale time.
* discount: Applied discount.

### Relationships

* order_id -> sales.orders.order_id
* product_id -> production.products.product_id

### Business Keywords

sales details, order details, line items, sold products, quantity sold, revenue, discount

### Typical Questions

* What are the best-selling products?
* What products generated the most revenue?
* What discounts were applied?
* How many units of each product were sold?

## Table: production.products

### Description

Stores product catalog information.

### Business Purpose

Represents products offered for sale. Used for product analysis, pricing analysis, category analysis, and brand performance.

### Columns

* product_id: Unique product identifier.
* product_name: Product name.
* brand_id: Product brand.
* category_id: Product category.
* model_year: Product model year.
* list_price: Standard product price.

### Relationships

* brand_id -> production.brands.brand_id
* category_id -> production.categories.category_id

### Business Keywords

product, item, merchandise, goods, inventory item, SKU, catalog

### Typical Questions

* What are the most expensive products?
* Which products sell the most?
* Which products belong to a category?
* What is the average product price?

## Table: production.brands

### Description

Stores product brand information.

### Business Purpose

Represents manufacturers or brands associated with products.

### Columns

* brand_id: Unique brand identifier.
* brand_name: Brand name.

### Relationships

* brand_id -> production.products.brand_id

### Business Keywords

brand, manufacturer, company, product brand, supplier brand

### Typical Questions

* Which brands generate the highest sales?
* How many products belong to each brand?
* What are the top-performing brands?

## Table: production.categories

### Description

Stores product category information.

### Business Purpose

Represents product groupings used for classification and reporting.

### Columns

* category_id: Unique category identifier.
* category_name: Category name.

### Relationships

* category_id -> production.products.category_id

### Business Keywords

category, product category, classification, product group

### Typical Questions

* Which category has the highest sales?
* How many products exist in each category?
* What are the most popular categories?

## Table: production.stocks

### Description

Stores inventory quantities for products in each store.

### Business Purpose

Represents current inventory levels and stock availability.

### Columns

* store_id: Store identifier.
* product_id: Product identifier.
* quantity: Available quantity.

### Relationships

* store_id -> sales.stores.store_id
* product_id -> production.products.product_id

### Business Keywords

inventory, stock, warehouse, availability, quantity on hand

### Typical Questions

* Which products are low in stock?
* What inventory is available per store?
* Which products are out of stock?

## Table: sales.stores

### Description

Stores information about physical store locations.

### Business Purpose

Represents sales locations and branches.

### Columns

* store_id: Unique store identifier.
* store_name: Store name.
* phone: Store phone.
* email: Store email.
* street: Address.
* city: City.
* state: State.
* zip_code: Postal code.

### Business Keywords

store, branch, location, shop, retail outlet

### Typical Questions

* Which store generates the most sales?
* How many stores exist?
* What are sales by store?

## Table: sales.staffs

### Description

Stores employee information.

### Business Purpose

Represents employees responsible for managing stores and orders.

### Columns

* staff_id: Unique employee identifier.
* first_name: Employee first name.
* last_name: Employee last name.
* email: Employee email.
* phone: Employee phone.
* active: Active status.
* store_id: Assigned store.
* manager_id: Manager identifier.

### Relationships

* store_id -> sales.stores.store_id
* manager_id -> sales.staffs.staff_id

### Business Keywords

employee, staff, salesperson, sales representative, manager

### Typical Questions

* Which employee generated the most sales?
* How many employees work in each store?
* Who reports to whom?
* What is employee performance by revenue?

## Business Relationship Paths

Customer Purchase Flow:

customers
→ orders
→ order_items
→ products

Product Hierarchy:

brands
→ products

categories
→ products

Sales Analysis Flow:

stores
→ orders
→ order_items

Employee Performance Flow:

staffs
→ orders
→ order_items

Inventory Flow:

stores
→ stocks
→ products
"""

In [2]:
from openai import OpenAI
BASE_URL = "https://api.gapgpt.app/v1"
API_KEY  = "sk-s8KnoW59PPxeHBvyzENeVoEiH2QbiNm1PxJt20H586up5p8n"

client_openai = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY
)

In [5]:
embedding_response = client_openai.embeddings.create(
    model="text-embedding-3-small",
    input=text
)

embedding = embedding_response.data[0].embedding

In [6]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

client = QdrantClient(url="http://localhost:6333")

client.recreate_collection(
    collection_name="schema_docs",
    vectors_config=VectorParams(
        size=len(embedding),
        distance=Distance.COSINE
    )
)

C:\Users\Aria\AppData\Local\Temp\ipykernel_14724\969735663.py:6: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [7]:
from qdrant_client.models import PointStruct
import uuid

client.upsert(
    collection_name="schema_docs",
    points=[
        PointStruct(
            id=str(uuid.uuid4()),
            vector=embedding,
            payload={
                "type": "schema",
                "content": text
            }
        )
    ]
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [8]:
question = "What are the best selling products?"

In [9]:
question_embedding = client_openai.embeddings.create(
    model="text-embedding-3-small",
    input=question
).data[0].embedding

In [11]:
results = client.query_points(
    collection_name="schema_docs",
    query=question_embedding,
    limit=3
)

In [12]:
results

QueryResponse(points=[ScoredPoint(id='badceb3f-d2d4-472d-b287-122e1546dd00', version=1, score=0.3334198, payload={'type': 'schema', 'content': '# BikeStore Database Knowledge Base\n\n## Table: sales.customers\n\n### Description\n\nStores customer information for people who purchase products from the store.\n\n### Business Purpose\n\nRepresents customers and their personal/contact information. Used to analyze customer behavior, purchase history, customer segmentation, and geographic distribution.\n\n### Columns\n\n* customer_id: Unique customer identifier.\n* first_name: Customer first name.\n* last_name: Customer last name.\n* phone: Customer phone number.\n* email: Customer email address.\n* street: Street address.\n* city: Customer city.\n* state: Customer state.\n* zip_code: Postal code.\n\n### Business Keywords\n\ncustomer, buyer, client, consumer, shopper, user, customer profile, customer information\n\n### Typical Questions\n\n* Who are the top customers by sales?\n* How many cus

In [13]:
customers_text = """Table: sales.customers

Description:
Stores information about customers who purchase products.

Business Purpose:
Represents buyers/clients of the store used for segmentation and customer analysis.

Columns:
- customer_id: unique customer identifier
- first_name: customer first name
- last_name: customer last name
- phone: contact number
- email: email address
- street: address street
- city: customer city
- state: customer state
- zip_code: postal code

Keywords:
customer, buyer, client, consumer, user, customer profile

Typical Questions:
- Who are top customers by sales?
- Which cities have most customers?
- How many customers do we have?"""

In [14]:
orders_text = """Table: sales.orders

Description:
Stores customer purchase orders and tracks order lifecycle.

Business Purpose:
Represents sales transactions between customers and store.

Columns:
- order_id: unique order id
- customer_id: customer reference
- order_status: status of order
- order_date: date order placed
- required_date: requested delivery date
- shipped_date: actual shipping date
- store_id: store handling order
- staff_id: employee handling order

Relationships:
- customer_id → sales.customers.customer_id
- store_id → sales.stores.store_id
- staff_id → sales.staffs.staff_id

Keywords:
order, purchase, sale, transaction, revenue, sales

Typical Questions:
- How many orders last month?
- Monthly sales trend?
- Orders by customer?
- Late shipments?"""

In [15]:
order_items_text = """Table: sales.order_items

Description:
Stores individual products inside each order (line items).

Business Purpose:
Used to calculate revenue, product performance, quantity sold, and discounts.

Columns:
- order_id: order reference
- item_id: line item id
- product_id: product reference
- quantity: number of items sold
- list_price: price at time of sale
- discount: applied discount

Relationships:
- order_id → sales.orders.order_id
- product_id → production.products.product_id

Keywords:
order items, line items, sales details, revenue, sold products

Typical Questions:
- Best selling products?
- Revenue per product?
- Total units sold?"""

In [16]:
products_text = """Table: production.products

Description:
Stores product catalog information.

Business Purpose:
Represents items available for sale and used for pricing and inventory analysis.

Columns:
- product_id
- product_name
- brand_id
- category_id
- model_year
- list_price

Relationships:
- brand_id → production.brands.brand_id
- category_id → production.categories.category_id

Keywords:
product, item, SKU, goods, merchandise

Typical Questions:
- Most expensive products?
- Products by category?
- Best selling products?"""

In [17]:
brands_text = """Table: production.brands

Description:
Stores product brand/manufacturer information.

Business Purpose:
Used to analyze brand performance and product grouping.

Columns:
- brand_id
- brand_name

Keywords:
brand, manufacturer, company

Typical Questions:
- Top performing brands?
- Sales by brand?"""

In [18]:
categories_text = """Table: production.categories

Description:
Stores product category classification.

Business Purpose:
Used for grouping products for reporting and analysis.

Columns:
- category_id
- category_name

Keywords:
category, product group, classification

Typical Questions:
- Best selling categories?
- Products per category?"""

In [20]:
stocks_text = """Table: production.stocks

Description:
Stores inventory quantity per product per store.

Business Purpose:
Used to track stock availability and inventory management.

Columns:
- store_id
- product_id
- quantity

Relationships:
- store_id → sales.stores.store_id
- product_id → production.products.product_id

Keywords:
inventory, stock, warehouse, availability

Typical Questions:
- Low stock products?
- Inventory per store?"""

In [21]:
stores_text = """Table: sales.stores

Description:
Stores physical store/branch information.

Business Purpose:
Used for regional sales analysis.

Columns:
- store_id
- store_name
- phone
- email
- street
- city
- state
- zip_code

Keywords:
store, branch, shop, location

Typical Questions:
- Sales by store?
- Best performing store?"""

In [22]:
staffs_text = """Table: sales.staffs

Description:
Stores employee information managing sales and stores.

Business Purpose:
Used for employee performance and sales attribution.

Columns:
- staff_id
- first_name
- last_name
- email
- phone
- active
- store_id
- manager_id

Relationships:
- store_id → sales.stores.store_id
- manager_id → sales.staffs.staff_id

Keywords:
staff, employee, salesperson, manager

Typical Questions:
- Top sales employees?
- Employee performance?"""

In [23]:
docs = [
    {"table": "sales.customers", "text": customers_text},
    {"table": "sales.orders", "text": orders_text},
    {"table": "sales.order_items", "text": order_items_text},
    {"table": "production.products", "text": products_text},
    {"table": "production.brands", "text": brands_text},
    {"table": "production.categories", "text": categories_text},
    {"table": "production.stocks", "text": stocks_text},
    {"table": "sales.stores", "text": stores_text},
    {"table": "sales.staffs", "text": staffs_text}
]

In [24]:
vectors = []

for doc in docs:
    emb = client_openai.embeddings.create(
        model="text-embedding-3-small",
        input=doc["text"]
    ).data[0].embedding

    vectors.append({
        "table": doc["table"],
        "text": doc["text"],
        "vector": emb
    })

In [25]:
from qdrant_client.models import PointStruct
import uuid

points = []

for v in vectors:
    points.append(
        PointStruct(
            id=str(uuid.uuid4()),
            vector=v["vector"],
            payload={
                "table": v["table"],
                "content": v["text"]
            }
        )
    )

client.upsert(
    collection_name="schema_docs",
    points=points
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [33]:
question = "Which brand has the most sales?"

In [34]:
question_embedding = client_openai.embeddings.create(
    model="text-embedding-3-small",
    input=question
).data[0].embedding

In [ ]:
# results = client.query_points(
#     collection_name="schema_docs",
#     query=question_embedding,
#     limit=3
# )

In [ ]:
# results

QueryResponse(points=[ScoredPoint(id='d8bd41b2-fd97-46d7-81df-3782dd02069e', version=3, score=0.37918764, payload={'table': 'production.products', 'content': 'Table: production.products\n\nDescription:\nStores product catalog information.\n\nBusiness Purpose:\nRepresents items available for sale and used for pricing and inventory analysis.\n\nColumns:\n- product_id\n- product_name\n- brand_id\n- category_id\n- model_year\n- list_price\n\nRelationships:\n- brand_id → production.brands.brand_id\n- category_id → production.categories.category_id\n\nKeywords:\nproduct, item, SKU, goods, merchandise\n\nTypical Questions:\n- Most expensive products?\n- Products by category?\n- Best selling products?'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='7008b5df-b5e8-4190-94d6-a84cb41da4e8', version=3, score=0.37655, payload={'table': 'sales.order_items', 'content': 'Table: sales.order_items\n\nDescription:\nStores individual products inside each order (line items).\n\nBusiness P

In [35]:
results = client.query_points(
    collection_name="schema_docs",
    query=question_embedding,
    limit=3
)

for p in results.points:
    print(f"{p.payload['table']} ({round(p.score, 2)})")

production.brands (0.38)
sales.stores (0.29)
sales.order_items (0.28)


In [39]:
import psycopg2


def get_connected_tables(table_name: str):

    sql = """
    SELECT
        tc.table_schema || '.' || tc.table_name AS source_table,
        ccu.table_schema || '.' || ccu.table_name AS target_table

    FROM information_schema.table_constraints tc

    JOIN information_schema.key_column_usage kcu
        ON tc.constraint_name = kcu.constraint_name
        AND tc.table_schema = kcu.table_schema

    JOIN information_schema.constraint_column_usage ccu
        ON ccu.constraint_name = tc.constraint_name
        AND ccu.table_schema = tc.table_schema

    WHERE tc.constraint_type = 'FOREIGN KEY'
    """

    conn = psycopg2.connect(
        host="localhost",
        database="bikestore",
        user="hamed",
        password="1234",
        port="5432"
    )

    cursor = conn.cursor()

    try:
        cursor.execute(sql)

        rows = cursor.fetchall()

        connected_tables = set()

        for source_table, target_table in rows:

            if source_table == table_name:
                connected_tables.add(target_table)

            elif target_table == table_name:
                connected_tables.add(source_table)

        return sorted(list(connected_tables))

    finally:
        cursor.close()
        conn.close()

In [42]:
get_connected_tables("production.brands")

['production.products']